In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    BooleanType
)

from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

CONFIG_TABLE = "weather_project.config.weather_locations"

# =====================================================
# 1. Create config schema if needed
# =====================================================

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS weather_project.config"
)

# =====================================================
# 2. Define desired location configuration
# =====================================================

location_records = [
    (
        "frisco_tx",
        "Frisco",
        "TX",
        "US",
        33.1507,
        -96.8236,
        1,
        True
    ),
    (
        "plano_tx",
        "Plano",
        "TX",
        "US",
        33.0198,
        -96.6989,
        2,
        True
    ),
    (
        "mckinney_tx",
        "McKinney",
        "TX",
        "US",
        33.1972,
        -96.6398,
        3,
        True
    ),
    (
        "prosper_tx",
        "Prosper",
        "TX",
        "US",
        33.2362,
        -96.8017,
        4,
        True
    ),
    (
        "little_elm_tx",
        "Little Elm",
        "TX",
        "US",
        33.1646,
        -96.9372,
        5,
        True
    )
]

location_schema = StructType([
    StructField("location_id", StringType(), False),
    StructField("city", StringType(), False),
    StructField("state_code", StringType(), False),
    StructField("country_code", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("request_order", IntegerType(), False),
    StructField("active", BooleanType(), False)
])

source_locations = (
    spark.createDataFrame(
        location_records,
        schema=location_schema
    )
    .withColumn(
        "updated_at",
        current_timestamp()
    )
)

In [0]:
if spark.catalog.tableExists(CONFIG_TABLE):

    target = DeltaTable.forName(
        spark,
        CONFIG_TABLE
    )

    (
        target.alias("target")
        .merge(
            source_locations.alias("source"),
            "target.location_id = source.location_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    (
        source_locations
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(CONFIG_TABLE)
    )

In [0]:
# check if schema exists
spark.sql(f"DESCRIBE SCHEMA EXTENDED weather_project.config").display()

# check if table exists
spark.sql(f"DESCRIBE EXTENDED weather_project.config.weather_locations").display()

# display table
spark.sql(f"SELECT * FROM weather_project.config.weather_locations").display()